# Langfuse Dataset Upload — DiabetesAssist

Builds a **correctness dataset** from ground-truth Q&A pairs and uploads it to
Langfuse so every future eval run scores against the same gold standard.

Each dataset item has:
- `input` — the question sent to the RAG system
- `expected_output` — the reference (gold) answer

**Required `.env` keys:**
```
LANGFUSE_PUBLIC_KEY=pk-lf-...
LANGFUSE_SECRET_KEY=sk-lf-...
LANGFUSE_HOST=https://cloud.langfuse.com   # optional, defaults to cloud
```

In [3]:
import sys, os
sys.path.append('..')

from dotenv import load_dotenv
load_dotenv('../.env')

from langfuse import Langfuse

lf = Langfuse(
    public_key=os.environ['LANGFUSE_PUBLIC_KEY'],
    secret_key=os.environ['LANGFUSE_SECRET_KEY'],
    host=os.environ.get('LANGFUSE_HOST', 'https://cloud.langfuse.com'),
)
print('Langfuse client ready')

Langfuse client ready


##  Define Ground-Truth Q&A Pairs

These are clinically reviewed reference answers drawn from the ADA Standards of Care
and the documents ingested into the vectorstore. Add or edit rows here to expand
the dataset over time. The colunm correct  is if answer is corrext  

The  dataset to evaluate the correctness of LLM responses on document-based QA tasks. It measures factual accuracy, retrieval quality, and hallucination detection using both correct and intentionally incorrect answers. The dataset was generated automatically using an LLM and Docling-extracted text/images, so it may contain biases, ambiguities, OCR errors, or hallucinated content. Human validation is recommended before production or clinical use.

In [5]:
import pandas as pd
df = pd.read_json('llm_eval_dataset.json')
df.head()

,question,answer,correct
0,What is the preferred insulin treatment for mo...,Continuous subcutaneous insulin infusion or mu...,1
1,Do insulin analogs reduce hypoglycemia risk co...,"Yes, insulin analogs are preferred because the...",1
2,What percentage reduction in microvascular com...,Approximately 50% reduction.,1
3,Was intensive insulin therapy associated with ...,"No, it was associated with higher severe hypog...",1
4,Can inhaled insulin be used instead of injecta...,"Yes, inhaled insulin may replace injectable pr...",1


Langfuse expeset a list of dictionaries with 'input' and 'expected_output' keys. We will convert our dataframe into that format.

In [6]:




CORRECTNESS_PAIRS = [
    {
        'input': {'question': row['question']},
        'expected_output': {'answer': row['answer']},
    }
    for _, row in df.iterrows()
]

print(f'{len(CORRECTNESS_PAIRS)} examples loaded')
df[['question', 'answer']].head()

20 examples loaded


,question,answer
0,What is the preferred insulin treatment for mo...,Continuous subcutaneous insulin infusion or mu...
1,Do insulin analogs reduce hypoglycemia risk co...,"Yes, insulin analogs are preferred because the..."
2,What percentage reduction in microvascular com...,Approximately 50% reduction.
3,Was intensive insulin therapy associated with ...,"No, it was associated with higher severe hypog..."
4,Can inhaled insulin be used instead of injecta...,"Yes, inhaled insulin may replace injectable pr..."


##  Create Dataset in Langfuse

`create_dataset` is idempotent — if the dataset name already exists Langfuse
returns the existing one rather than raising an error.


In [7]:
DATASET_NAME = 'medical-rag-correctness_1'

lf.create_dataset(
    name=DATASET_NAME,
    description='Ground-truth Q&A pairs for DiabetesAssist RAG correctness evaluation',
    metadata={'source': 'ADA Standards of Care / ingested PDFs'},
)
print(f"Dataset '{DATASET_NAME}' ready")

Dataset 'medical-rag-correctness_1' ready


You can view the evaluation results directly in the Langfuse UI :![alt text](img/langfuse_dataset.png)

##  Upload Items

Each call to `create_dataset_item` adds one Q&A pair. Items are deduplicated by
`id` if you pass one — useful for re-running without duplicating rows.

In [8]:
for item in CORRECTNESS_PAIRS:
    lf.create_dataset_item(
        dataset_name=DATASET_NAME,
        input=item['input'],
        expected_output=item['expected_output'],
    )

lf.flush()
print(f'Uploaded {len(CORRECTNESS_PAIRS)} items to "{DATASET_NAME}"')

Uploaded 20 items to "medical-rag-correctness_1"


we can see the dataset and items in the Langfuse dashboard now. Next, we will set up an evaluation that runs the DiabetesAssist RAG agent on these questions and compares the agent's answers to the expected answers we just uploaded. [alt text](img/langfuse_dataset.png)

## Verify Upload

Fetch items back from Langfuse to confirm they landed.

In [9]:
dataset = lf.get_dataset(DATASET_NAME)
print(f'{len(dataset.items)} items in "{DATASET_NAME}":\n')
for item in dataset.items:
    print(f'  [{item.id}]')
    print(f'    input : {item.input["question"][:70]}')
    print(f'    output: {item.expected_output["answer"][:70]}\n')

20 items in "medical-rag-correctness_1":

  [96e95247-1ff8-4523-b770-e42839572e83]
    input : What condition is also known as long COVID?
    output: Post COVID-19 condition (PCC).

  [0e0c6dcb-b558-4c2a-9a4b-62b07d26ee7c]
    input : What percentage of symptomatic COVID-19 patients developed severe dise
    output: Approximately 15%.

  [bb8f8e17-8abb-466e-a6a4-f28e5374b819]
    input : Does WHO strongly recommend against awake prone positioning in COVID-1
    output: Yes, WHO strongly recommends against it.

  [0a71f347-759a-4ebb-937d-fcfeebfcf334]
    input : What is the recommendation for awake prone positioning in severe COVID
    output: WHO conditionally recommends awake prone positioning for severely ill 

  [045a1b56-9591-4598-97d0-f0768571258e]
    input : What organization published the Clinical Management of COVID-19 living
    output: The World Health Organization (WHO).

  [3a674db8-f050-480e-a532-5377b11bd703]
    input : What HbA1c target is recommended for diabetes ma